# Day 23 — Project: Cost Center Variance Analysis
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Replicate KSB1/S_ALR_87013611-style cost center reporting in Python
2. Calculate absolute and percentage variances by period and cost element
3. Use pandas Styler for conditional formatting of a variance report

---
> **SAP Context:** KSB1 shows actual line items per cost center; S_ALR_87013611 is the classic CO-OM variance report (actual vs plan). Fields: KOSTL (cost center), KSTAR (cost element), GJAHR (fiscal year), PERIOD (posting period 1-12), ACTUAL_AMT, PLAN_AMT.


## 0. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
cc = pd.read_csv(DATA / 'cost_center_actuals.csv')
print(cc.shape)
print(cc['GJAHR'].unique(), cc['PERIOD'].unique())
cc.head(4)

## 1. Calculate Variance and Variance % by Period

In [ ]:
# YOUR SOLUTION
# Variance = ACTUAL_AMT - PLAN_AMT (positive = over budget)
# Variance % = (ACTUAL_AMT - PLAN_AMT) / PLAN_AMT * 100

cc['VARIANCE'] = cc['ACTUAL_AMT'] - cc['PLAN_AMT']
cc['VARIANCE_PCT'] = np.where(
    cc['PLAN_AMT'] != 0,
    (cc['VARIANCE'] / cc['PLAN_AMT']) * 100,
    np.nan
)

# Period-level summary
period_summary = (
    cc.groupby(['GJAHR', 'PERIOD'])
    .agg(ACTUAL=('ACTUAL_AMT','sum'), PLAN=('PLAN_AMT','sum'), VARIANCE=('VARIANCE','sum'))
    .assign(VAR_PCT=lambda d: (d['VARIANCE'] / d['PLAN'] * 100).round(1))
    .reset_index()
)
print(period_summary.to_string(index=False))

## 2. Top 5 Over-Budget and Under-Budget Cost Elements per Period

In [ ]:
# YOUR SOLUTION
# Focus on a specific fiscal year
YEAR = 2024

cc_yr = cc[cc['GJAHR'] == YEAR].copy()

# Aggregate by period + cost element
ce_period = (
    cc_yr.groupby(['PERIOD', 'KSTAR', 'KSTAR_DESC'])
    .agg(ACTUAL=('ACTUAL_AMT','sum'), PLAN=('PLAN_AMT','sum'), VARIANCE=('VARIANCE','sum'))
    .reset_index()
)

# For a target period (e.g. period 6)
TARGET_PERIOD = 6
ce_p = ce_period[ce_period['PERIOD'] == TARGET_PERIOD]

top5_over = ce_p.nlargest(5, 'VARIANCE')[['KSTAR','KSTAR_DESC','ACTUAL','PLAN','VARIANCE']]
top5_under = ce_p.nsmallest(5, 'VARIANCE')[['KSTAR','KSTAR_DESC','ACTUAL','PLAN','VARIANCE']]

print(f'=== Period {TARGET_PERIOD} — Top 5 OVER Budget ===')
print(top5_over.to_string(index=False))
print(f'\n=== Period {TARGET_PERIOD} — Top 5 UNDER Budget ===')
print(top5_under.to_string(index=False))

## 3. YTD Actual vs YTD Plan

In [ ]:
# YOUR SOLUTION
YTD_THROUGH_PERIOD = 9  # e.g., report through September

ytd = cc[(cc['GJAHR'] == YEAR) & (cc['PERIOD'] <= YTD_THROUGH_PERIOD)]

ytd_by_cc = (
    ytd.groupby(['KOSTL', 'KTEXT'])
    .agg(YTD_ACTUAL=('ACTUAL_AMT','sum'), YTD_PLAN=('PLAN_AMT','sum'))
    .assign(YTD_VARIANCE=lambda d: d['YTD_ACTUAL'] - d['YTD_PLAN'],
            YTD_VAR_PCT=lambda d: (d['YTD_ACTUAL'] - d['YTD_PLAN']) / d['YTD_PLAN'] * 100)
    .reset_index()
    .sort_values('YTD_VARIANCE', ascending=False)
)

print(f'=== YTD (Periods 1-{YTD_THROUGH_PERIOD}) by Cost Center ===')
print(ytd_by_cc.to_string(index=False))

## 4. Rolling 12-Period Trend for a Specific Cost Center

In [ ]:
# YOUR SOLUTION
TARGET_CC = cc['KOSTL'].iloc[0]   # pick first cost center

cc_trend = (
    cc[cc['KOSTL'] == TARGET_CC]
    .groupby(['GJAHR', 'PERIOD'])
    .agg(ACTUAL=('ACTUAL_AMT','sum'), PLAN=('PLAN_AMT','sum'))
    .reset_index()
    .sort_values(['GJAHR', 'PERIOD'])
    .tail(12)   # last 12 periods
    .reset_index(drop=True)
)

cc_trend['PERIOD_LABEL'] = cc_trend['GJAHR'].astype(str) + '.' + cc_trend['PERIOD'].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cc_trend['PERIOD_LABEL'], cc_trend['ACTUAL'], marker='o', label='Actual', color='#e74c3c', linewidth=2)
ax.plot(cc_trend['PERIOD_LABEL'], cc_trend['PLAN'], marker='s', label='Plan', color='#3498db',
        linewidth=2, linestyle='--')
ax.fill_between(range(len(cc_trend)),
                cc_trend['ACTUAL'].values, cc_trend['PLAN'].values,
                alpha=0.15, color='red', label='Variance (shaded)')
ax.set_title(f'Cost Center {TARGET_CC} — Rolling 12-Period Actual vs Plan', fontweight='bold')
ax.set_ylabel('Amount (USD)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('day23_cc_trend.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Variance Report with Conditional Highlighting (pandas Styler)

In [ ]:
# YOUR SOLUTION
# Conditional highlighting: red if variance > 5%, green if under budget by >5%

def color_variance(val):
    if pd.isna(val):
        return ''
    if val > 5:
        return 'background-color: #ffcccc; color: #900'
    elif val < -5:
        return 'background-color: #ccffcc; color: #090'
    else:
        return 'background-color: #fff9cc'

report_df = ytd_by_cc[['KOSTL','KTEXT','YTD_ACTUAL','YTD_PLAN','YTD_VARIANCE','YTD_VAR_PCT']].copy()
report_df.columns = ['Cost Center','Description','YTD Actual','YTD Plan','YTD Variance','Var %']

styled = (
    report_df.style
    .applymap(color_variance, subset=['Var %'])
    .format({'YTD Actual': '${:,.0f}', 'YTD Plan': '${:,.0f}',
             'YTD Variance': '${:,.0f}', 'Var %': '{:.1f}%'})
    .set_caption('YTD Cost Center Variance Report — Conditional Highlighting')
    .hide(axis='index')
)
styled

## Daily Prompt
**Answer independently:**

1. The CFO asks: "Which single cost element is responsible for the most budget overrun YTD?" Write the query to answer this.
2. In SAP S_ALR_87013611, you can drill from a cost center down to individual cost element line items. Replicate this drill-down in Python: write a function `drill_down(kostl, period)` that returns the line-item detail for a given cost center and period.
3. What is the difference between KSTAR (cost element) and KTEXT/KSTAR_DESC in SAP CO? Why does it matter for reporting?
